# 1.8.1 Sample Splitting


Sample splitting is a technique used to evaluate the performance of predictive models by separating the data into two independent parts to prevent being overly optimistic about the performence of a model which overfits to the training data.

In practice, the data is randomly split into a train and a test sample, the model is fitted on the training set, and predictions generated on the test set. These are compared with the observed outcomes of the test set using metrics like MSE or R^2. Comparing different model specifications can help evaluate which model generalizes better ot unseen data.

In a possible extension, one can stratify based on certain features to ensure that the samples are comparable in these dimensions.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

In [2]:
# loading the data
file = "https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/main/data/wage2015_subsample_inference.csv"
df = pd.read_csv(file)

df.describe()

,wage,lwage,sex,shs,hsg,scl,clg,ad,mw,so,we,ne,exp1,exp2,exp3,exp4,occ,occ2,ind,ind2
count,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000
mean,23.410410,2.970787,0.444466,0.023301,0.243883,0.278058,0.317670,0.137087,0.259612,0.296505,0.216117,0.227767,13.760583,3.018925,8.235867,25.118038,5310.737476,11.670874,6629.154951,13.316893
std,21.003016,0.570385,0.496955,0.150872,0.429465,0.448086,0.465616,0.343973,0.438464,0.456761,0.411635,0.419432,10.609465,4.000904,14.488962,53.530225,11874.356080,6.966684,5333.443992,5.701019
min,3.021978,1.105912,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,10.000000,1.000000,370.000000,2.000000
25%,13.461538,2.599837,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,0.250000,0.125000,0.062500,1740.000000,5.000000,4880.000000,9.000000
50%,19.230769,2.956512,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,10.000000,1.000000,1.000000,1.000000,4040.000000,13.000000,7370.000000,14.000000
75%,27.777778,3.324236,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,21.000000,4.410000,9.261000,19.448100,5610.000000,17.000000,8190.000000,18.000000
max,528.845673,6.270697,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,47.000000,22.090000,103.823000,487.968100,100000.000000,22.000000,100000.000000,22.000000


In [3]:
# splitting the data into train and test sets
train, test = train_test_split(df.drop('wage', axis=1), test_size=0.20, random_state=123, stratify=df['sex'])
train.shape, test.shape

((4120, 19), (1030, 19))

Here the data was split in a ratio 80/20 for train/test data and additionally it was stratified by sex which means that the split is performed by sex to generate a balanced sample along this dimension. In our case this is done for instructive purposes as it is unlikely that one data set might end up with significantly more samples of one category.

In [4]:
# 1. Basic Model
model_base = 'lwage ~ sex + exp1 + shs + hsg + scl + clg + mw + so + we + C(occ2) + C(ind2)'
base = smf.ols(model_base, data=train)
results_base = base.fit()

# in-sample performance
rsquared_base = results_base.rsquared
rsquared_adj_base = results_base.rsquared_adj
mse_base = np.mean(results_base.resid**2)
mse_adj_base = results_base.mse_resid

# out-of-sample performance
y_pred_base_test = results_base.predict(test.drop('lwage', axis=1))
mse_base_test = np.mean((test['lwage'] - y_pred_base_test) ** 2)
rsquared_base_test = 1 - (np.sum((test['lwage'] - y_pred_base_test) ** 2) / np.sum((test['lwage'] - test['lwage'].mean()) ** 2))

# results
print(f'Rsquared={rsquared_base:.4f}')
print(f'Rsquared_adjusted={rsquared_adj_base:.4f}')
print(f'Rsquared_test={rsquared_base_test:.4f}')
print(f'MSE={mse_base:.4f}')
print(f'MSE_adjusted={mse_adj_base:.4f}')
print(f'MSE_test={mse_base_test:.4f}')

Rsquared=0.3165
Rsquared_adjusted=0.3081
Rsquared_test=0.2767
MSE=0.2233
MSE_adjusted=0.2261
MSE_test=0.2311


This model generalizes reasonably well to the unseen test data as R^2 is only slightly lower in the test sample and MSE increases by only a bit. Moreover the adjusted values do not deviate by too much.

In [5]:
# 2. Overfitting Model
model_flex = ('lwage ~ sex + shs+hsg+scl+clg+C(occ2)+C(ind2)+mw+so+we '
              '+ (exp1+exp2+exp3+exp4)*(shs+hsg+scl+clg+C(occ2)+C(ind2)+mw+so+we)'
              '+ (exp1+exp2+exp3+exp4)**2*(shs+hsg+scl+clg)'
              '+ (exp1+exp2+exp3+exp4)*(shs+hsg+scl+clg)**2')
flex = smf.ols(model_flex, data=train.sample(frac=0.5, random_state=123))
results_flex = flex.fit()

# in-sample performance
rsquared_flex = results_flex.rsquared
rsquared_adj_flex = results_flex.rsquared_adj
mse_flex = np.mean(results_flex.resid**2)
mse_adj_flex = results_flex.mse_resid

# out-of-sample performance
y_pred_flex_test = results_flex.predict(test.drop('lwage', axis=1))
mse_flex_test = np.mean((test['lwage'] - y_pred_flex_test) ** 2)
rsquared_flex_test = 1 - (np.sum((test['lwage'] - y_pred_flex_test) ** 2) / np.sum((test['lwage'] - test['lwage'].mean()) ** 2))    

print(f'Rsquared={rsquared_flex:.4f}')
print(f'Rsquared_adjusted={rsquared_adj_flex:.4f}')
print(f'Rsquared_test={rsquared_flex_test:.4f}')
print(f'MSE={mse_flex:.4f}')
print(f'MSE_adjusted={mse_adj_flex:.4f}')
print(f'MSE_test={mse_flex_test:.4f}')

Rsquared=0.4235
Rsquared_adjusted=0.3409
Rsquared_test=0.1203
MSE=0.1876
MSE_adjusted=0.2145
MSE_test=0.2811


Fitting a more complex model to only half of the train data (i.e. reducing the sample size while increasing the number of variables), the in-sample metrics look promising but evaluating the prediction on the test sample clearly shows that the model overfits the training data.